# CSV Basics — 10: Streaming CSV with Structured Streaming

## What you will learn
Structured Streaming can process CSV files as they arrive in a directory.
This is useful for legacy systems that export CSV files on a schedule.

In this notebook:
1. File-based streaming source for CSV
2. Schema enforcement in streaming
3. `maxFilesPerTrigger` and processing rate control
4. Error handling in streaming CSV
5. Archiving processed files
6. Monitoring streaming progress


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 19:46:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


In [2]:
import pathlib, shutil, threading, time, random, datetime, json as pyjson

# Setup directories
STREAM_INPUT  = f"{DATA_DIR}/stream_csv_input"
STREAM_OUTPUT = f"{DATA_DIR}/stream_csv_output"
STREAM_CKPT   = f"{DATA_DIR}/stream_csv_checkpoint"
STREAM_ARCHIVE= f"{DATA_DIR}/stream_csv_archive"

for d in [STREAM_INPUT, STREAM_OUTPUT, STREAM_CKPT, STREAM_ARCHIVE]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Streaming directories ready:")
print(f"  Input    : {STREAM_INPUT}   (drop CSV files here)")
print(f"  Output   : {STREAM_OUTPUT}  (processed Parquet)")
print(f"  Checkpoint: {STREAM_CKPT}")

Streaming directories ready:
  Input    : /workspace/data/csv_basics/stream_csv_input   (drop CSV files here)
  Output   : /workspace/data/csv_basics/stream_csv_output  (processed Parquet)
  Checkpoint: /workspace/data/csv_basics/stream_csv_checkpoint


## Step 1 — Define Schema and Streaming Reader

In streaming, schema MUST be defined explicitly.
`inferSchema` is not supported for streaming DataFrames.


In [3]:
event_schema = StructType([
    StructField("event_id",         LongType()),
    StructField("user_id",          LongType()),
    StructField("event_type",       StringType()),
    StructField("page",             StringType()),
    StructField("revenue",          DoubleType()),
    StructField("event_date",       DateType()),
    StructField("_corrupt_record",  StringType()),  # MUST be in schema for PERMISSIVE mode
])

# Streaming reader — watches STREAM_INPUT for new CSV files
csv_stream = (
    spark.readStream
         .format("csv")
         .schema(event_schema)
         .option("header", True)
         .option("maxFilesPerTrigger", 1)
         .option("ignoreLeadingWhiteSpace", True)
         .option("ignoreTrailingWhiteSpace", True)
         .option("mode", "PERMISSIVE")
         .option("columnNameOfCorruptRecord", "_corrupt_record")
         .load(STREAM_INPUT)
)

print(f"Is streaming: {csv_stream.isStreaming}")
print("Schema:")
csv_stream.printSchema()

Is streaming: True
Schema:
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- event_date: date (nullable = true)
 |-- _corrupt_record: string (nullable = true)



## Step 2 — Write Stream to Parquet Output


In [4]:
# Process: filter good rows, add processing timestamp
processed = (
    csv_stream
    .filter(F.col("_corrupt_record").isNull())  # only good rows
    .drop("_corrupt_record")
    .withColumn("_processed_at", F.current_timestamp())
    .withColumn("_source_date",  F.current_date())
)

# Write to Parquet (partitioned by source date)
query = (
    processed
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CKPT)
    .option("path", STREAM_OUTPUT)
    .partitionBy("_source_date")
    .queryName("csv_stream_processor")
    .start()
)

print(f"Streaming query started: {query.name}")
print(f"Status: {query.status['message']}")

26/04/10 19:46:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming query started: csv_stream_processor
Status: Initializing sources


## Step 3 — Generate CSV Files to Stream


In [5]:
EVENTS    = ["page_view","click","purchase","search","add_to_cart"]
PAGES     = ["/home","/products","/cart","/checkout"]

def generate_csv_batch(batch_id, n=50):
    """Write a CSV file to the streaming input directory."""
    base = datetime.date(2024, 1, 1)
    rows = ["event_id,user_id,event_type,page,revenue,event_date"]
    for i in range(n):
        d = base + datetime.timedelta(days=random.randint(0, 90))
        rev = round(random.uniform(10, 500), 2) if random.random() > 0.7 else 0.0
        rows.append(
            f"{batch_id*1000+i},{random.randint(1,500)},"
            f"{random.choice(EVENTS)},{random.choice(PAGES)},"
            f"{rev},{d}"
        )
    path = f"{STREAM_INPUT}/events_{batch_id:04d}.csv"
    pathlib.Path(path).write_text("\n".join(rows))
    return path

print("Generating CSV batches for streaming...")
for i in range(1, 5):
    path = generate_csv_batch(i)
    print(f"  Dropped: {path.split('/')[-1]}")
    time.sleep(3)

time.sleep(5)
print("\nStream processed. Checking output...")

if pathlib.Path(STREAM_OUTPUT).exists():
    output_count = spark.read.parquet(STREAM_OUTPUT).count()
    print(f"  Output rows: {output_count:,}")
else:
    print("  Output not yet written")

Generating CSV batches for streaming...
  Dropped: events_0001.csv


[Stage 0:>                                                          (0 + 1) / 1]

  Dropped: events_0002.csv


  Dropped: events_0003.csv


  Dropped: events_0004.csv

Stream processed. Checking output...
  Output rows: 200


## Step 4 — Monitor Streaming Progress


In [6]:
# Check last progress
if query.lastProgress:
    prog = query.lastProgress
    print("Last batch progress:")
    print(f"  Batch ID           : {prog.get('batchId')}")
    print(f"  Input rows         : {prog.get('numInputRows')}")
    print(f"  Rows/sec           : {prog.get('processedRowsPerSecond', 0):.1f}")
    print(f"  Trigger duration   : {prog.get('durationMs', {}).get('triggerExecution', 0)} ms")
    print()
    print(f"  Source (files seen) : {prog.get('sources', [{}])[0].get('numInputRows', 0)}")

print("\nAll progress batches:")
for p in query.recentProgress[-3:]:
    print(f"  batch={p.get('batchId')} rows={p.get('numInputRows')} "
          f"rate={p.get('processedRowsPerSecond',0):.0f}/s")

query.stop()
print("\nStream stopped.")

Last batch progress:
  Batch ID           : 3
  Input rows         : 50
  Rows/sec           : 47.8
  Trigger duration   : 1044 ms

  Source (files seen) : 50

All progress batches:
  batch=1 rows=50 rate=46/s
  batch=2 rows=50 rate=19/s
  batch=3 rows=50 rate=48/s

Stream stopped.


26/04/10 19:47:18 WARN DAGScheduler: Failed to cancel job group 22c045de-c3f1-4e16-af3e-ef6304579301. Cannot find active jobs for it.
26/04/10 19:47:18 WARN DAGScheduler: Failed to cancel job group 22c045de-c3f1-4e16-af3e-ef6304579301. Cannot find active jobs for it.


## Step 5 — Final Output Verification


In [7]:
# Verify output
print("Streaming pipeline output:")
try:
    df_output = spark.read.parquet(STREAM_OUTPUT)
    print(f"  Total rows processed: {df_output.count():,}")
    print()
    print("  Distribution by event type:")
    df_output.groupBy("event_type").count().orderBy(F.desc("count")).show()
    print("  Revenue summary:")
    df_output.filter(F.col("revenue") > 0) \
             .agg(F.count("*").alias("purchase_events"),
                  F.sum("revenue").alias("total_revenue"),
                  F.avg("revenue").alias("avg_revenue")).show()
except Exception as e:
    print(f"  No output yet: {e}")

Streaming pipeline output:
  Total rows processed: 200

  Distribution by event type:
+-----------+-----+
| event_type|count|
+-----------+-----+
|      click|   42|
|   purchase|   42|
|     search|   41|
|add_to_cart|   39|
|  page_view|   36|
+-----------+-----+

  Revenue summary:
+---------------+------------------+------------------+
|purchase_events|     total_revenue|       avg_revenue|
+---------------+------------------+------------------+
|             60|12610.439999999999|210.17399999999998|
+---------------+------------------+------------------+



## Summary: Streaming CSV Pattern

```python
# Define schema ALWAYS (inferSchema not supported in streaming)
schema = StructType([...])

# Streaming reader
stream = (
    spark.readStream
         .format("csv")
         .schema(schema)
         .option("header", True)
         .option("maxFilesPerTrigger", 1)     # rate control
         .option("mode", "PERMISSIVE")
         .load("/path/to/input/dir")
)

# Write to Parquet
query = (
    stream.writeStream
          .format("parquet")
          .outputMode("append")
          .option("checkpointLocation", "/path/checkpoint")
          .option("path", "/path/output")
          .start()
)
```

### maxFilesPerTrigger tuning
- `1` — conservative, low latency, high trigger overhead
- `5–10` — balanced for most cases
- `1000` — batch-like, process large backlog quickly

### Production considerations
- Always use `columnNameOfCorruptRecord` for bad row capture
- Monitor `numInputRows` — if 0 for many batches, check input path
- Checkpoint enables exactly-once — never delete it while stream is running
